<a href="https://colab.research.google.com/github/christinengalle19-collab/Assignement2New/blob/main/Assignment_13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##For this assignment, I selected a text dataset from Project Gutenberg. It provides public domain books that are legally accessible and easy to preprocess. The dataset is lightweight, suitable for training a simple generative model, and allows clear evaluation of text coherence, vocabulary patterns, and stylistic reproduction.

##Description of GPTs architecture and its functionality.
##GPT (Generative Pre trained Transformer) is a deep learning architecture based on the Transformer model, which uses layers of self attention and feed forward neural networks to process and generate text. It is trained on vast amounts of text data to learn patterns of language, grammar, and context. Functionally, GPT predicts the next word in a sequence by analyzing the relationships between all previous words, enabling it to produce coherent, context aware responses. Its architecture allows it to handle long range dependencies in text, making it powerful for tasks like translation, summarization, and conversational AI.

In [19]:
!pip install transformers torch
!pip install requests
!pip install torch
!pip install transformers


In [20]:
import numpy as np

import requests
import re
from transformers import pipeline, set_seed
import torch
import tensorflow as tf
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM
from keras.layers import Embedding

In [21]:
#load the text from gutenberg project
url = "https://www.gutenberg.org/files/11/11-0.txt"
response = requests.get(url)
text = response.text
print(text[:500])

*** START OF THE PROJECT GUTENBERG EBOOK 11 ***

[Illustration]




Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

 CHAPTER I.     Down the Rabbit-Hole
 CHAPTER II.    The Pool of Tears
 CHAPTER III.   A Caucus-Race and a Long Tale
 CHAPTER IV.    The Rabbit Sends in a Little Bill
 CHAPTER V.     Advice from a Caterpillar
 CHAPTER VI.    Pig and Pepper
 CHAPTER VII.   A Mad Tea-Party
 CHAPTER VIII.  The Queen’s Croquet-Ground
 CHAPTER IX.    The


In [22]:
#clean the text byremoving header and footer
text = re.sub(r'\[.*?\]', '', text)
text = text.strip()
print(text[:500])


*** START OF THE PROJECT GUTENBERG EBOOK 11 ***






Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

 CHAPTER I.     Down the Rabbit-Hole
 CHAPTER II.    The Pool of Tears
 CHAPTER III.   A Caucus-Race and a Long Tale
 CHAPTER IV.    The Rabbit Sends in a Little Bill
 CHAPTER V.     Advice from a Caterpillar
 CHAPTER VI.    Pig and Pepper
 CHAPTER VII.   A Mad Tea-Party
 CHAPTER VIII.  The Queen’s Croquet-Ground
 CHAPTER IX.    The Mock Turtle’s


In [23]:
#save the text in a file
with open("shakespeare.txt", "w", encoding="utf-8") as file:
    file.write(text)
    print("Text saved to shakespeare.txt")

Text saved to shakespeare.txt


In [24]:
## Load the pre-trained model and tokenizer
#Next, we'll load a pre-trained GPT-2 model and its associated tokenizer. The tokenizer is responsible for converting text into tokens that the model can understand.

model_name = "gpt2"
model = GPT2LMHeadModel.from_pretrained(model_name)
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [25]:
#tokenizing the text
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_index = len(tokenizer.word_index) + 1
print(total_index)

3066


In [26]:
# Analyze the vocabulary
#top 20 frequent words
sorted_words = sorted(tokenizer.word_counts.items(), key=lambda x: x[1], reverse=True)
top_words = sorted_words[:20]
print("Top 20 frequent words:")
for word, count in top_words:
    print(f"{word}: {count}")
    print("\n")


Top 20 frequent words:
the: 1623


”: 1040


and: 795


to: 719


a: 621


she: 535


of: 502


it: 494


said: 459


alice: 385


in: 361


was: 356


you: 319


i: 273


that: 264


as: 254


her: 248


at: 206


on: 192


with: 179




In [27]:
#split for training
sequences = []
for line in text.split('\n'):
    tokens = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(tokens)):
        n_gram_sequence = tokens[:i+1]
sequences = tokenizer.texts_to_sequences([text])
sequences = np.array(sequences)
max_length = max([len(seq) for seq in sequences])
sequences = pad_sequences(sequences, maxlen=max_length, padding='pre')
x = sequences[:, :-1]
y = sequences[:, -1]
y = to_categorical(y, num_classes=total_index)
print(x.shape)
print(y.shape)


(1, 27763)
(1, 3066)


In [28]:
#Modele LSTM
model = Sequential()
model.add(Embedding(total_index, 10, input_length=max_length-1))
model.add(LSTM(50))
model.add(Dense(total_index, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(x, y, epochs=100, verbose=1)


Epoch 1/100


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 10s 10s/step - accuracy: 0.0000e+00 - loss: 8.0289
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 1.0000 - loss: 8.0253
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 1.0000 - loss: 8.0218
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 1.0000 - loss: 8.0182
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 1.0000 - loss: 8.0144
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 1.0000 - loss: 8.0103
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 1.0000 - loss: 8.0057
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - accuracy: 1.0000 - loss: 8.0005
Epoch 9/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step - accuracy: 1.0000 - loss: 7.9947
Epoch 10/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 1.0000 - loss: 7.9878
Epoch 11/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 1.0000 - loss: 7.9798
Epoch 12/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 1.0000 - loss: 7.9701
Epoch 13/100
1/1 ━━

In [29]:
## Define a function for text generation
#This function will be our main tool for generating text. It takes a prompt and several parameters that control the generation process.


def generate_text(prompt, max_length=100, temperature=0.7, num_return_sequences=1, top_k=50, top_p=0.95):
    # Encode the prompt and get attention mask
    inputs = tokenizer(prompt, return_tensors='pt', padding=True)
    input_ids = inputs['input_ids']
    attention_mask = inputs['attention_mask']

    output = model.generate(
        input_ids,
        attention_mask=attention_mask,
        max_length=max_length,
        temperature=temperature,
        num_return_sequences=num_return_sequences,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True, # Enable sampling for temperature to take effect
        top_k=top_k, # Pass top_k parameter
        top_p=top_p # Pass top_p parameter
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [30]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# Save the current Keras tokenizer and model instances
original_keras_tokenizer = tokenizer
original_keras_model = model

# Re-initialize the GPT2Tokenizer and GPT2LMHeadModel for the generate_text function
# This ensures the correct tokenizer and model are used for the generation process.
model_name = "gpt2"
openai_tokenizer = GPT2Tokenizer.from_pretrained(model_name)
openai_model = GPT2LMHeadModel.from_pretrained(model_name)

# Set the pad token for the GPT2Tokenizer
# This is necessary because the tokenizer call with `padding=True` requires a pad token.
openai_tokenizer.pad_token = openai_tokenizer.eos_token

# Temporarily assign the GPT2Tokenizer and GPT2LMHeadModel to the global variables
tokenizer = openai_tokenizer
model = openai_model

# Call the generate function
#prompt 1
generated_text_output = generate_text('Hello Word')
print(generated_text_output)

# Restore the Keras tokenizer and model to the global variables
# to avoid breaking other parts of the notebook that might rely on them.
tokenizer = original_keras_tokenizer
model = original_keras_model

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Hello Word

Folks, if you have a question about any of these topics, you might want to read the answer to this one.

If you've made it this far and are still curious, here are some things I learned from the original article on the subject.

What is the relationship between an e-mail and the "real world"?

I first discovered this topic when I was a kid. I was just one of many kids who got their e-


In [31]:
# Save the current global Keras tokenizer and model instances
# These variables ('original_keras_tokenizer', 'original_keras_model') should exist from cell 'aAA4J2n8BS4f'.
# No need to re-save them as they represent the original Keras objects.

# Temporarily assign the GPT2Tokenizer and GPT2LMHeadModel (loaded in aAA4J2n8BS4f)
# to the global 'tokenizer' and 'model' variables for this function call.
# We assume 'openai_tokenizer' and 'openai_model' were correctly defined and 'openai_tokenizer.pad_token' was set in 'aAA4J2n8BS4f'.
temp_saved_tokenizer = tokenizer
temp_saved_model = model

tokenizer = openai_tokenizer
model = openai_model

# Call the generate function
#prompt2
generated_text_output = generate_text('I really like ai')
print(generated_text_output)

# Restore the original global tokenizer and model (Keras instances)
tokenizer = temp_saved_tokenizer
model = temp_saved_model

I really like ai."

"That's a lot of words. It's just a matter of having a good time."

"I don't really know about you. I'd like to know about you."

"I'm sorry, but I don't want to talk about you."

"I'm sorry. I'm very sorry."

"Don't be rude to me. I'll take care of that."

"No, no. I


In [32]:
# Temporarily assign the GPT2Tokenizer and GPT2LMHeadModel (loaded in aAA4J2n8BS4f)
# to the global 'tokenizer' and 'model' variables for this function call.
# We assume 'openai_tokenizer' and 'openai_model' were correctly defined and 'openai_tokenizer.pad_token' was set in 'aAA4J2n8BS4f'.
temp_saved_tokenizer = tokenizer
temp_saved_model = model

tokenizer = openai_tokenizer
model = openai_model

# Call the generate function
prompt3= generate_text('willis college is the place to do BIA program')
print("Exercise 3: Adjusting temperature")
print("Low temperature (0.2):")
print(generate_text(prompt3, temperature=0.2, max_length=200)) # Increased max_length
print("\nHigh temperature (1.5):")
print(generate_text(prompt3, temperature=1.5, max_length=200)) # Increased max_length
print("\n" + "="*50 + "\n")

# Restore the original global tokenizer and model (Keras instances)
tokenizer = temp_saved_tokenizer
model = temp_saved_model

Exercise 3: Adjusting temperature
Low temperature (0.2):
willis college is the place to do BIA program.

"I have a lot of questions," she said. "What is it about this that I don't want to hear? I don't want to hear what my students think. I don't want to hear what my students think. I want to hear what my students think. So, I'm looking at it as a way to give them a sense of what I think the program is for."

But the college has a long history of being a place for students to learn.

"We have a lot of great programs that are in the BIA program," said Dr. Robert H. Dickson, director of the BIA program at the University of North Carolina at Chapel Hill. "We have a lot of great programs that are in the BIA program. We have a lot of great programs that are in the BIA program."

Dickson said the college has been a place for students to

High temperature (1.5):
willis college is the place to do BIA program.

"I have a lot of questions," she said. "What is it about this that I don't want to h

In [33]:
# Temporarily assign the GPT2Tokenizer and GPT2LMHeadModel (loaded in aAA4J2n8BS4f)
# to the global 'tokenizer' and 'model' variables for this function call.
# We assume 'openai_tokenizer' and 'openai_model' were correctly defined and 'openai_tokenizer.pad_token' was set in 'aAA4J2n8BS4f'.
temp_saved_tokenizer = tokenizer
temp_saved_model = model

tokenizer = openai_tokenizer
model = openai_model
#call the generation functiom
prompt4 = generate_text("In the year 2050, transportation will")
print("Exercise 4: Using top_k and top_p")
print("Default settings:")
print(generate_text(prompt4, max_length=200)) # Increased max_length
print("\nLow top_k and top_p:")
print(generate_text(prompt4, top_k=10, top_p=0.5, max_length=200)) # Increased max_length
print("\n" + "="*50 + "\n")


Exercise 4: Using top_k and top_p
Default settings:
In the year 2050, transportation will be the dominant mode of transportation. The transportation system is expected to reach its capacity by 2040.

In the years 2050 and 2060, transportation will be the predominant mode of transportation. The transportation system is expected to reach its capacity by 2040.

In the years 2050 and 2060, transportation will be the predominant mode of transportation. The transportation system is expected to reach its capacity by 2040.

In the years 2050 and 2060, transportation will be the predominant mode of transportation. The transportation system is expected to reach its capacity by 2040.

In the years 2050 and 2060, transportation will be the predominant mode of transportation. The transportation system is expected to reach its capacity by 2040.

In the years 2050 and 2060, transportation will be the predominant mode of transportation. The transportation system is expected to reach its capacity by 20

In [34]:
# Temporarily assign the GPT2Tokenizer and GPT2LMHeadModel (loaded in aAA4J2n8BS4f)
# to the global 'tokenizer' and 'model' variables for this function call.
# We assume 'openai_tokenizer' and 'openai_model' were correctly defined and 'openai_tokenizer.pad_token' was set in 'aAA4J2n8BS4f'.
temp_saved_tokenizer = tokenizer
temp_saved_model = model

tokenizer = openai_tokenizer
model = openai_model
#call the generation functiom

#Exercise 5
prompt5= generate_text('AI system will always needs human intervention')
print("Exercise 5: Prompt engineering")
base_prompt = "Write a short story about"
topics = ["a robot learning to paint", "a time traveler's first day in the past", "a sentient AI's ethical dilemma"]

for topic in topics:
    full_prompt = f"{base_prompt} {topic}. The story should be creative and engaging:"
    print(f"Topic: {topic}")
    print(generate_text(full_prompt, max_length=200))
    print("\n" + "-"*150 + "\n")

Exercise 5: Prompt engineering
Topic: a robot learning to paint
Write a short story about a robot learning to paint. The story should be creative and engaging:

This is a story about a robot learning to paint. The story should be creative and engaging: The story should be short and very short. The story should be short and very short. The story should be short and very short. The story should be short and very short. The story should be short and very short. The story should be short and very short. The story should be short and very short. The story should be short and very short. The story should be short and very short. The story should be short and very short. The story should be short and very short. The story should be short and very short. The story should be short and very short. The story should be short and very short. The story should be short and very short. The story should be short and very short. The story should be short and very short. The story should be short and ver